In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

In [7]:
class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size)
        )

    def forward(self, x):
        return self.model(x)

    def fit(self, X, y, epochs=100, lr=0.001):
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.parameters(), lr=lr)

        self.train()

        for epoch in range(epochs):
            # Forward pass
            outputs = self(X)
            loss = criterion(outputs, y)

            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

    def predict(self, X):
        self.eval()

        with torch.no_grad():
            outputs = self(X)
            predictions = torch.argmax(outputs, dim=1)

        return predictions

In [10]:
X = torch.randn(64, 10)
y = torch.randint(0, 2, (64,))

model = MLP(input_size=10, hidden_size=32, output_size=2)
criterion = nn.CrossEntropyLoss()
model.eval()
model.zero_grad()
outputs = model(X)
loss = criterion(outputs, y)

loss.backward()
full_grads = [
    p.grad.clone()
    for p in model.parameters()
]
model.zero_grad()

for i in range(0, 64, 16):
    X_batch = X[i:i+16]
    y_batch = y[i:i+16]
    outputs = model(X_batch)
    loss = criterion(outputs, y_batch)
    loss = loss / 4
    loss.backward()
mini_grads = [
    p.grad.clone()
    for p in model.parameters()
]
max_diff = max(
    (g1 - g2).abs().max().item()
    for g1, g2 in zip(full_grads, mini_grads)
)
print("Maximum gradient difference:", max_diff)

Maximum gradient difference: 5.587935447692871e-09
